In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from mlflow.models.signature import infer_signature
from dotenv import load_dotenv

# 1. Force reload environment variables
load_dotenv(Path("..") / ".env", override=True)

# 2. Extract and validate critical configuration parameters
tracking_url = os.getenv("MLFLOW_TRACKING_URL")
username = "xoxeb69158@heavty.com"
password =  "Gauravkumar007"

# Pull S3 / Object Storage keys
aws_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret = os.getenv("AWS_SECRET_ACCESS_KEY")

if not tracking_url:
    raise ValueError("❌ MLFLOW_TRACKING_URL is missing from the configuration environment.")
if not aws_key or not aws_secret:
    raise ValueError("❌ Object storage access keys are missing. MLflow needs these to upload artifacts.")

# Apply to tracking server variables
mlflow.set_tracking_uri(tracking_url)
os.environ["MLFLOW_TRACKING_USERNAME"] = username
os.environ["MLFLOW_TRACKING_PASSWORD"] = password

# Expose S3 credentials and set the exact required endpoint
os.environ["AWS_ACCESS_KEY_ID"] = aws_key
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://team18.ipstorage.tatacommunications.com"

print("🛰️ Track server authentication injected successfully.")
print("📦 Storage system credentials bound to context.")
print("🔗 S3 Endpoint URL target set to:", os.environ["MLFLOW_S3_ENDPOINT_URL"])

# 3. Define the experiment space for RailOps
mlflow.set_experiment("railops_predictive_maintenance")

print("🛰️ Loading track dataset from local workspace...")
df = pd.read_csv("/home/jovyan/move-it/01_dataset/downloaded_railway_telemetry.csv")

# --- DEFENSIVE DATA PREPROCESSING GUARD ---
# Separate categorical text/identifiers from numeric ML training inputs
numeric_cols = ['vibration_hz', 'acoustic_emission_db', 'rail_strain_mu', 'track_temperature_c', 'alignment_deviation_mm', 'last_maintenance_days']
available_numeric = [col for col in numeric_cols if col in df.columns]

if not available_numeric:
    # Fallback to choosing any available numeric elements dynamically if layout names mismatch
    available_numeric = list(df.select_dtypes(include=[np.number]).columns)
    if 'failure_probability' in available_numeric:
        available_numeric.remove('failure_probability')

# Ensure the target vector is discrete/integers if using a Classifier
if 'failure_probability' in df.columns:
    # Convert probabilities to a binary target class (1 if prob > 0.5 else 0) to fit a Classifier model
    if df['failure_probability'].dtype == 'float64' or df['failure_probability'].max() <= 1.0:
        y = (df['failure_probability'] > 0.5).astype(int)
    else:
        y = df['failure_probability'].astype(int)
else:
    raise KeyError("❌ Target column 'failure_probability' missing from source dataset configuration.")

X = df[available_numeric].fillna(0) # Fill missing numbers to prevent matrix fitting crashes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize MLflow Run Context
with mlflow.start_run(run_name="RandomForest_Baseline"):
    print("🤖 Training predictive maintenance model...")
    
    # Model parameters
    n_estimators = 100
    max_depth = 6
    
    # Log hyperparameters to MLflow dashboard tracking interface
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    # Build and fit classifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate performance
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"✅ Training complete. Model Accuracy: {accuracy:.4f}")
    
    # Log evaluation performance metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # 5. Log the structural asset feature importances
    for feature_name, importance in zip(X.columns, model.feature_importances_):
        mlflow.log_metric(f"importance_{feature_name}", importance)
        
    print("Tracking URI:", mlflow.get_tracking_uri())
    print("Endpoint:", os.environ.get("MLFLOW_S3_ENDPOINT_URL"))
    print("Access Key:", os.environ.get("AWS_ACCESS_KEY_ID"))
    
    # 6. Register and serialize the model asset catalog item
    signature = infer_signature(X_train, model.predict(X_train))
    print("X_train columns:")
    print(X_train.columns)
    print(signature)
    
    # Log to MLflow Tracking Server
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="railops_model",
        signature=signature,
        input_example=X_train.iloc[:5],
        registered_model_name="RailOps_Maintenance_Model"
    )
    print("🎉 Done! Model tracking parameters and artifact payloads successfully pushed to MLflow.")

    # 7. Local Artifact Generation Guard (Required for upload_model.py)
    model_dir = Path("../04_starter-kit")
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "model.joblib"

    joblib.dump(model, model_path)
    print(f"💾 Local pipeline validation secure. Artifact saved to: {model_path}")

🛰️ Track server authentication injected successfully.
📦 Storage system credentials bound to context.
🔗 S3 Endpoint URL target set to: https://team18.ipstorage.tatacommunications.com
🛰️ Loading track dataset from local workspace...
🤖 Training predictive maintenance model...
✅ Training complete. Model Accuracy: 0.8050


/home/jovyan/move-it/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/13 08:55:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Tracking URI: https://xoxeb-mlflow-9c83d-paas-7250-22326-public.cloudservices.tatacommunications.com
Endpoint: https://team18.ipstorage.tatacommunications.com
Access Key: AKIA376XU41JS037X0OW
X_train columns:
Index(['vibration_hz', 'acoustic_emission_db', 'rail_strain_mu',
       'track_temperature_c', 'alignment_deviation_mm',
       'last_maintenance_days'],
      dtype='object')
inputs: 
  ['vibration_hz': double (required), 'acoustic_emission_db': double (required), 'rail_strain_mu': double (required), 'track_temperature_c': double (required), 'alignment_deviation_mm': double (required), 'last_maintenance_days': long (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



Registered model 'RailOps_Maintenance_Model' already exists. Creating a new version of this model...
2026/07/13 08:55:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RailOps_Maintenance_Model, version 15
Created version '15' of model 'RailOps_Maintenance_Model'.


🎉 Done! Model tracking parameters and artifact payloads successfully pushed to MLflow.
💾 Local pipeline validation secure. Artifact saved to: ../04_starter-kit/model.joblib
🏃 View run RandomForest_Baseline at: https://xoxeb-mlflow-9c83d-paas-7250-22326-public.cloudservices.tatacommunications.com/#/experiments/1/runs/14f0c776fddc4df7b1ced295858a5317
🧪 View experiment at: https://xoxeb-mlflow-9c83d-paas-7250-22326-public.cloudservices.tatacommunications.com/#/experiments/1
